# Altry sunucusu (ücretsiz Google Colab GPU)

Bu defter Altry'nin yapay zekâ sunucusunu Colab'ın ücretsiz ekran kartında çalıştırır ve telefonundaki
uygulamanın bağlanacağı adresi verir.

1. Menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç.
2. Aşağıdaki iki hücreyi sırayla çalıştır.
3. En altta çıkan **adresi ve erişim anahtarını** uygulamadaki *Ayarlar* ekranına yaz.

Colab oturumu kapanınca sunucu da kapanır; yeniden açtığında adres değişir.

In [ ]:
#@title 1) Kurulum (ilk seferde birkaç dakika sürer)
REPO = "https://github.com/hotllock/altry"  #@param {type:"string"}
BRANCH = ""  #@param {type:"string"}
import os, subprocess
if not os.path.exists("/content/altry"):
    args = ["git", "clone", "--depth", "1"] + (["-b", BRANCH] if BRANCH else []) + [REPO, "/content/altry"]
    subprocess.run(args, check=True)
else:  # already installed: fetch the latest version
    subprocess.run(["git", "-C", "/content/altry", "pull", "--ff-only"])
!apt-get -qq install -y ffmpeg fonts-dejavu-core > /dev/null
!pip install -q -r /content/altry/server/requirements.txt demucs
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK - Çalışma zamanı türünü T4 GPU yap")

In [ ]:
#@title 2) Sunucuyu başlat
import os, re, secrets, subprocess, time
token = secrets.token_urlsafe(9)
env = dict(os.environ, ALTRY_TOKEN=token, ALTRY_DATA="/content/altry-data")
server = subprocess.Popen(["python", "-m", "uvicorn", "altry.app:app", "--host", "127.0.0.1", "--port", "8000"],
                          cwd="/content/altry/server", env=env)
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        url = found.group(0)
        break
print("\n" + "=" * 60)
print("Sunucu adresi   :", url)
print("Erişim anahtarı :", token)
print("=" * 60)
print("Bu hücre çalıştığı sürece sunucu açıktır. Durdurmak için hücreyi durdur.")
try:
    server.wait()
finally:
    tunnel.terminate(); server.terminate()